In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv('orders.csv')
order_products = pd.read_csv('order_products_prior.csv')
products = pd.read_csv('products.csv')
departments = pd.read_csv('departments.csv')

In [ ]:
df = order_products.merge(orders, on='order_id')
df = df.merge(products, on='product_id')
df = df.merge(departments, on='department_id')

In [ ]:
# cart size (items per order)
cart_size = df.groupby('order_id').size().reset_index(name='cart_size')
df = df.merge(cart_size, on='order_id')

# reordered count per order
reordered_count = df.groupby('order_id')['reordered'].sum().reset_index(name='reordered_count')
df = df.merge(reordered_count, on='order_id')

# reorder rate per order
df['reorder_rate'] = df['reordered'] / df['cart_size']

# weekend indicator
df['is_weekend'] = df['order_dow'].apply(lambda x: 1 if x in [0,6] else 0)

In [ ]:
df.to_csv('clean_instacart.csv', index=False)

In [1]:
#line graph showing average reorder rate for different cart sizes.
r_by_size = order_level.groupby('cart_size')['reorder_rate'].mean().reset_index().sort_values('cart_size')
max_size = 20
r_by_size_plot = r_by_size[r_by_size['cart_size'] <= max_size]

plt.figure()
plt.plot(r_by_size_plot['cart_size'], r_by_size_plot['reorder_rate'], marker='o', color='blue', label='Avg reorder rate')
plt.title('How does average reorder rate change with cart size?')
plt.xlabel('Cart size (items per order)')
plt.ylabel('Fraction of items in order that were reordered)')
plt.xticks(r_by_size_plot['cart_size'])
plt.ylim(0,1)
plt.grid(True)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [ ]:
#histogram showing the frequency of different cart sizes.
plt.figure()
hist_max = 40
cart_vals = order_level['cart_size'].clip(upper=hist_max)
bins = np.arange(1, hist_max+2) - 0.5
plt.hist(cart_vals, bins=bins, edgecolor='black', color='pink')
plt.title('Distribution of cart sizes across orders of cart sizes')
plt.xlabel(f'Cart size (items per order). Values >= {hist_max} are clipped for display')
plt.ylabel('Number of orders ')
plt.xticks(range(1, hist_max+1, 2))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# bar chart showing the average reorder rate by department.
dept_stats = df.groupby('department')['reordered'].mean().rename('avg_reorder_rate').reset_index()
dept_counts = df.groupby('department').size().rename('n_items').reset_index()
dept_stats = dept_stats.merge(dept_counts, on='department').sort_values('avg_reorder_rate', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(dept_stats['department'], dept_stats['avg_reorder_rate'], color='purple', alpha=0.9)
plt.gca().invert_yaxis()
plt.title('Which departments have highest average item-level reorder rates?')
plt.xlabel('Fraction of items in that department reordered')
plt.ylabel('Department name')
plt.xlim(0,1)

for i, (rate, n) in enumerate(zip(dept_stats['avg_reorder_rate'], dept_stats['n_items'])):
plt.text(rate + 0.01, i, f'n={n}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
#bar chart showing reorder counts for the top products.

prod_reorders = df.groupby(['product_id','product_name'])['reordered'].sum().rename('reorder_count').reset_index()
top10 = prod_reorders.sort_values('reorder_count', ascending=False).head(10)

plt.figure(figsize=(10,6))
y = top10['product_name'][::-1]
x = top10['reorder_count'][::-1]
plt.barh(y, x, color='green')
plt.title('Top 10 most frequently reordered products')
plt.xlabel('Number of times product was reordered)')
plt.ylabel('Product name')

for xi, yi in zip(x, y):
plt.text(xi + max(x)*0.01, yi, f'{int(xi)}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
#line graph showing total orders by hour.

orders_by_hour = order_level.groupby('order_hour_of_day').size().rename('n_orders').reset_index().sort_values('order_hour_of_day')

plt.figure()
plt.plot(orders_by_hour['order_hour_of_day'], orders_by_hour['n_orders'], marker='o', linestyle='-', color='orange', label='Total orders')
plt.title('How many orders occur at each hour of day?')
plt.xlabel('Hour of day (0-23)')
plt.ylabel('Number of orders')
plt.xticks(range(0,24))
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#line graph showing reorder rate by hour.
hour_reorder = order_level.groupby('order_hour_of_day')['reorder_rate'].mean().reset_index().sort_values('order_hour_of_day')

plt.figure()
plt.plot(hour_reorder['order_hour_of_day'], hour_reorder['reorder_rate'], marker='o', linestyle='-', color='crimson', label='Avg reorder rate')
plt.title('Do reorder rates differ by hour of day?')
plt.xlabel('Hour of day (0-23)')
plt.ylabel('Fraction of items in order that were reordered)')
plt.xticks(range(0,24))
plt.ylim(0,1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()